<div align="center">

# Heavy Equipment Selling Price Prediction

### Notebook v17: Fast.ai Structured Category Encoding & Grandmaster GBDT Ensembling

---

**Student Name:** _Priyanshu Raj_  
**Student ID:** _24F2008980_  
**Course:** _Machine Learning Practices Project_  
**Competition:** Heavy Equipment Selling Price Prediction Challenge  

---

</div>


## Table of Contents

1. [Problem Statement & Historical Kaggle Benchmark Insights](#1)
2. [Dataset Overview](#2)
3. [Import Libraries](#3)
4. [Load Dataset](#4)
5. [Exploratory Data Analysis](#5)
6. [Fast.ai Preprocessing & Sentinel Normalization](#6)
7. [Comprehensive Date-Time Feature Extraction](#7)
8. [Multi-Percentile Out-of-Fold Target Encodings](#8)
9. [Level-1 Base Models Training (10-Fold Stratified CV)](#9)
10. [Level-2 Non-Negative SLSQP Meta-Learner](#10)
11. [Model Performance Comparison & Feature Importance](#11)
12. [Final Predictions & Export](#12)
13. [Conclusion & Key Takeaways](#13)


<a id="1"></a>
# 1. Problem Statement & Historical Kaggle Benchmark Insights

The goal is to predict heavy equipment selling prices (`TargetValue`). The metric is **Root Mean Squared Logarithmic Error (RMSLE)**, evaluated as **RMSE on `log1p(TargetValue)`**.

### Historical Insights from "Blue Book for Bulldozers":
1. **Category Integer Codes**: Tree-based models perform best when categoricals are converted into ordinal integer codes preserving category hierarchy.
2. **Explicit Date Deconstruction**: Splitting dates into `Year`, `Month`, `Day`, `DayOfWeek`, `DayOfYear`, `ElapsedDays`, and `Quarter` allows tree models to split temporal trends effectively.
3. **Pure GBDT Ensembling**: Linear models struggle on tabular spec data (~0.297 RMSLE). Ensembling **LightGBM**, **CatBoost**, **XGBoost**, and **ExtraTrees** gives optimal non-linear precision.



<a id="2"></a>
# 2. Dataset Overview

- **Train Set:** 138,701 rows × 50 columns.
- **Test Set:** 15,000 rows × 49 columns.
- **Features:** Equipment attributes, operational hours, year of manufacture, region codes, vendor IDs, col1..col30 spec flags, and text descriptors.


<a id="3"></a>
# 3. Import Libraries


In [ ]:
import os
import re
import time
import warnings
from scipy.optimize import minimize

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

print("✅ Fast.ai Notebook v17 Libraries Loaded Successfully!")


<a id="4"></a>
# 4. Load Dataset


In [ ]:
data_dir = r"e:\HeavyEquipmentML\data"
target_dir_root = "/kaggle/input"

if os.path.exists(target_dir_root):
    for dirpath, _, filenames in os.walk(target_dir_root):
        if "train.csv" in filenames:
            data_dir = dirpath
            break
elif os.path.exists("data"):
    data_dir = "data"

print(f"📂 Data Directory Path: {data_dir}")

df_train_raw = pd.read_csv(os.path.join(data_dir, "train.csv"), low_memory=False)
df_test_raw = pd.read_csv(os.path.join(data_dir, "test.csv"), low_memory=False)
sub_template = pd.read_csv(os.path.join(data_dir, "sample_submission.csv"))

print(f"📊 Raw Train Shape : {df_train_raw.shape[0]:,} rows × {df_train_raw.shape[1]} columns")
print(f"📊 Raw Test Shape  : {df_test_raw.shape[0]:,} rows × {df_test_raw.shape[1]} columns")
print(f"📊 Submission Shape: {sub_template.shape[0]:,} rows × {sub_template.shape[1]} columns")



<a id="5"></a>
# 5. Exploratory Data Analysis (EDA)


In [ ]:
log_target_val = np.log1p(df_train_raw["TargetValue"].clip(lower=0))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df_train_raw["TargetValue"], kde=True, ax=axes[0], color="royalblue", bins=50)
axes[0].set_title(f"Original Target Value (Skew: {df_train_raw['TargetValue'].skew():.2f})")

sns.histplot(log_target_val, kde=True, ax=axes[1], color="teal", bins=50)
axes[1].set_title(f"log1p Transformed Target (Skew: {log_target_val.skew():.2f})")
plt.tight_layout()
plt.show()


<a id="6"></a>
# 6. Fast.ai Preprocessing & Comprehensive Feature Engineering


In [ ]:
def add_datepart(df, fldname, drop=True, time=False):
    fld = pd.to_datetime(df[fldname])
    targ_pre = re.sub('[aA]lready', '', fldname)
    attr = ['Year', 'Month', 'Week', 'Day', 'Dayofweek', 'Dayofyear',
            'Is_month_end', 'Is_month_start', 'Is_quarter_end', 'Is_quarter_start', 'Is_year_end', 'Is_year_start']
    if time:
        attr = attr + ['Hour', 'Minute', 'Second']
    for k in attr:
        if k == 'Week':
            df[targ_pre + k] = fld.dt.isocalendar().week.astype(int)
        else:
            df[targ_pre + k] = getattr(fld.dt, k.lower())
    df[targ_pre + 'Elapsed'] = (fld - pd.Timestamp('1990-01-01')).dt.days
    if drop:
        df.drop(columns=[fldname], inplace=True, errors="ignore")
    return df

def proc_df_fastai(train_df, test_df):
    tr = train_df.copy()
    te = test_df.copy()

    drop_cols = ["InventoryGroupDescription", "TransactionID"]
    tr.drop(columns=drop_cols, inplace=True, errors="ignore")
    te.drop(columns=drop_cols, inplace=True, errors="ignore")

    for col in ["YearMade", "ManufactureYear"]:
        if col in tr.columns:
            tr.loc[tr[col] == 1001, col] = np.nan
            te.loc[te[col] == 1001, col] = np.nan

    hours_col = "MeterHours" if "MeterHours" in tr.columns else "OperationalHoursMeter"
    for df in (tr, te):
        df["Hours_na"] = df[hours_col].isna().astype(int)
        df[hours_col] = df[hours_col].fillna(0)
        df["Hours_log"] = np.log1p(df[hours_col])
        if hours_col != "Hours_log":
            df.drop(columns=[hours_col], inplace=True, errors="ignore")

    mfg_col = "YearMade" if "YearMade" in tr.columns else "ManufactureYear"
    
    tr = add_datepart(tr, "TransactionDate")
    te = add_datepart(te, "TransactionDate")

    full_df = pd.concat([tr.assign(is_train=1), te.assign(is_train=0)], ignore_index=True)

    year_made = pd.to_numeric(full_df[mfg_col], errors="coerce").fillna(2000)
    full_df["MachineAge"] = (full_df["TransactionDateYear"] - year_made).clip(lower=0)
    full_df["MachineAge_Sq"] = full_df["MachineAge"] ** 2

    # Fast.ai Style Categorical Processing: Convert strings to Categorical codes
    cat_cols = full_df.select_dtypes(include=["object"]).columns.tolist()
    for col in cat_cols:
        full_df[col] = full_df[col].astype("category").cat.codes + 1

    df_tr_out = full_df[full_df["is_train"] == 1].drop(columns=["is_train"])
    df_te_out = full_df[full_df["is_train"] == 0].drop(columns=["is_train"])

    return df_tr_out, df_te_out

df_tr_fe, df_te_fe = proc_df_fastai(train_prepped if 'train_prepped' in locals() else df_train_raw, test_prepped if 'test_prepped' in locals() else df_test_raw)
print(f"✅ Fast.ai Preprocessing Complete: Train {df_tr_fe.shape}, Test {df_te_fe.shape}")


<a id="8"></a>
# 8. Multi-Percentile Out-of-Fold Target Encodings


In [ ]:
TARGET_ENCODE_VARS = [
    "Spec_BaseClass", "Spec_SubClass", "RegionCode", "VendorPartnerID",
    "ProductConfigID"
]

def generate_multi_percentile_oof_te(train_df, test_df, target_col="TargetValue", n_splits=10, smoothing=20.0):
    tr = train_df.copy()
    te = test_df.copy()

    y_log = np.log1p(tr[target_col].clip(lower=0))
    g_mean = y_log.mean()
    g_median = y_log.median()
    g_q25 = y_log.quantile(0.25)
    g_q75 = y_log.quantile(0.75)

    bins = pd.qcut(y_log, q=10, labels=False, duplicates="drop")
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    if "AssetID" in tr.columns:
        tr["AssetID_OOF_Price"] = g_mean
        te["AssetID_OOF_Price"] = g_mean
        for trn_idx, val_idx in skf.split(tr, bins):
            asset_map = tr.iloc[trn_idx].groupby("AssetID")[target_col].apply(lambda s: np.log1p(s.clip(lower=0)).mean()).to_dict()
            tr.loc[tr.index[val_idx], "AssetID_OOF_Price"] = tr.iloc[val_idx]["AssetID"].map(asset_map).fillna(g_mean).values
        full_asset_map = tr.groupby("AssetID")[target_col].apply(lambda s: np.log1p(s.clip(lower=0)).mean()).to_dict()
        te["AssetID_OOF_Price"] = te["AssetID"].map(full_asset_map).fillna(g_mean)

    for col in TARGET_ENCODE_VARS:
        if col not in tr.columns:
            continue

        m_col = f"{col}_TE_Mean"
        med_col = f"{col}_TE_Median"

        tr[m_col], tr[med_col] = g_mean, g_median
        te[m_col], te[med_col] = g_mean, g_median

        for trn_idx, val_idx in skf.split(tr, bins):
            tr_sub = tr.iloc[trn_idx].copy()
            val_sub = tr.iloc[val_idx].copy()

            tr_sub["target_log"] = np.log1p(tr_sub[target_col].clip(lower=0))
            stats = tr_sub.groupby(col)["target_log"].agg(
                count="count",
                mean="mean",
                median="median"
            ).reset_index()

            stats["smooth_mean"] = (stats["count"] * stats["mean"] + smoothing * g_mean) / (stats["count"] + smoothing)

            mean_map = stats.set_index(col)["smooth_mean"].to_dict()
            med_map = stats.set_index(col)["median"].to_dict()

            tr.loc[tr.index[val_idx], m_col] = val_sub[col].map(mean_map).fillna(g_mean).values
            tr.loc[tr.index[val_idx], med_col] = val_sub[col].map(med_map).fillna(g_median).values

        full_sub = tr.copy()
        full_sub["target_log"] = np.log1p(full_sub[target_col].clip(lower=0))
        full_stats = full_sub.groupby(col)["target_log"].agg(
            count="count",
            mean="mean",
            median="median"
        ).reset_index()

        full_stats["smooth_mean"] = (full_stats["count"] * full_stats["mean"] + smoothing * g_mean) / (full_stats["count"] + smoothing)

        full_mean_map = full_stats.set_index(col)["smooth_mean"].to_dict()
        full_med_map = full_stats.set_index(col)["median"].to_dict()

        te[m_col] = te[col].map(full_mean_map).fillna(g_mean)
        te[med_col] = te[col].map(full_med_map).fillna(g_median)

    return tr, te

train_te, test_te = generate_multi_percentile_oof_te(df_tr_fe, df_te_fe)
print("✅ Out-of-Fold Target Encodings Generated!")


<a id="9"></a>
# 9. Level-1 Base Models Training (10-Fold CV across 4 Tree Model Families)


In [ ]:
drop_cols = ["SalesID", "TransactionDate", "TargetValue", "AssetID"]
feature_cols = [c for c in train_te.columns if c not in drop_cols]

X = train_te[feature_cols].copy().fillna(-999).astype(np.float32)
y_log = np.log1p(train_te["TargetValue"].clip(lower=0))
X_test = test_te[feature_cols].copy().fillna(-999).astype(np.float32)

print(f"📊 Feature Matrix Assembled: {X.shape[1]} features")

N_SPLITS = 10
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
target_bins = pd.qcut(y_log, q=10, labels=False, duplicates="drop")

oof_lgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))
oof_xgb = np.zeros(len(X))
oof_et = np.zeros(len(X))

test_lgb = np.zeros(len(X_test))
test_cat = np.zeros(len(X_test))
test_xgb = np.zeros(len(X_test))
test_et = np.zeros(len(X_test))

# Hyperparameters tuned for Fast.ai Integer Category Encodings
lgb_params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.015,
    "num_leaves": 128,
    "max_depth": 10,
    "min_child_samples": 25,
    "subsample": 0.8,
    "colsample_bytree": 0.65,
    "random_state": 42,
    "n_estimators": 4500,
    "verbose": -1,
    "n_jobs": -1
}

cat_params = {
    "loss_function": "RMSE",
    "eval_metric": "RMSE",
    "learning_rate": 0.02,
    "depth": 8,
    "l2_leaf_reg": 4,
    "subsample": 0.8,
    "random_seed": 42,
    "iterations": 4000,
    "verbose": False,
    "thread_count": -1
}

xgb_params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "learning_rate": 0.015,
    "max_depth": 8,
    "subsample": 0.8,
    "colsample_bytree": 0.65,
    "random_state": 42,
    "n_estimators": 4000,
    "tree_method": "hist",
    "n_jobs": -1
}

et_params = {
    "n_estimators": 120,
    "max_depth": 24,
    "min_samples_split": 4,
    "n_jobs": -1,
    "random_state": 42
}

print(f"=== Training Level-1 Base Models ({N_SPLITS}-Fold CV) ===")
for fold, (trn_idx, val_idx) in enumerate(skf.split(X, target_bins), start=1):
    f_start = time.time()
    X_tr, y_tr = X.iloc[trn_idx], y_log.iloc[trn_idx]
    X_va, y_va = X.iloc[val_idx], y_log.iloc[val_idx]

    # LightGBM
    m_lgb = lgb.LGBMRegressor(**lgb_params)
    m_lgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(150, verbose=False)])
    oof_lgb[val_idx] = m_lgb.predict(X_va)
    test_lgb += m_lgb.predict(X_test) / N_SPLITS

    # CatBoost
    m_cat = CatBoostRegressor(**cat_params)
    m_cat.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=150, verbose=False)
    oof_cat[val_idx] = m_cat.predict(X_va)
    test_cat += m_cat.predict(X_test) / N_SPLITS

    # XGBoost
    m_xgb = xgb.XGBRegressor(**xgb_params)
    m_xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    oof_xgb[val_idx] = m_xgb.predict(X_va)
    test_xgb += m_xgb.predict(X_test) / N_SPLITS

    # ExtraTrees
    m_et = ExtraTreesRegressor(**et_params)
    m_et.fit(X_tr, y_tr)
    oof_et[val_idx] = m_et.predict(X_va)
    test_et += m_et.predict(X_test) / N_SPLITS

    print(f"Fold {fold}/{N_SPLITS} Finished in {time.time() - f_start:.1f}s")

lgb_rmsle = np.sqrt(mean_squared_error(y_log, oof_lgb))
cat_rmsle = np.sqrt(mean_squared_error(y_log, oof_cat))
xgb_rmsle = np.sqrt(mean_squared_error(y_log, oof_xgb))
et_rmsle = np.sqrt(mean_squared_error(y_log, oof_et))

print(f"✅ LightGBM   10-Fold OOF RMSLE: {lgb_rmsle:.5f}")
print(f"✅ CatBoost   10-Fold OOF RMSLE: {cat_rmsle:.5f}")
print(f"✅ XGBoost    10-Fold OOF RMSLE: {xgb_rmsle:.5f}")
print(f"✅ ExtraTrees 10-Fold OOF RMSLE: {et_rmsle:.5f}")


<a id="10"></a>
# 10. Level-2 Non-Negative SLSQP Meta Blending


In [ ]:
print("=== Training Level-2 Non-Negative Constrained Meta-Learner ===")

OOF_matrix = np.column_stack([oof_lgb, oof_cat, oof_xgb, oof_et])
Test_matrix = np.column_stack([test_lgb, test_cat, test_xgb, test_et])

def loss_func(weights):
    w = weights / np.sum(weights)
    blend_oof = OOF_matrix @ w
    return np.sqrt(mean_squared_error(y_log, blend_oof))

init_w = np.array([0.40, 0.35, 0.20, 0.05])
bounds = [(0, 1)] * 4
res = minimize(loss_func, init_w, method="SLSQP", bounds=bounds)

opt_weights = res.x / np.sum(res.x)

oof_opt_blend = OOF_matrix @ opt_weights
test_preds_opt_blend = Test_matrix @ opt_weights

masterpiece_rmsle = np.sqrt(mean_squared_error(y_log, oof_opt_blend))

actual_y = np.expm1(y_log)
actual_pred = np.expm1(oof_opt_blend)
dollar_rmse = np.sqrt(mean_squared_error(actual_y, actual_pred))
dollar_mae = mean_absolute_error(actual_y, actual_pred)

print(f"🔥 SLSQP Non-Negative Optimal Weights : {opt_weights}")
print(f"🔥 V17 FAST.AI ENSEMBLE OOF RMSLE     : {masterpiece_rmsle:.5f}")
print(f"📊 Dollar-Scale Actual Metrics          : RMSE = ${dollar_rmse:,.2f} | MAE = ${dollar_mae:,.2f}")


<a id="11"></a>
# 11. Model Performance Comparison


In [ ]:
results_df = pd.DataFrame([
    {"Model": "LightGBM", "OOF RMSLE": lgb_rmsle},
    {"Model": "CatBoost", "OOF RMSLE": cat_rmsle},
    {"Model": "XGBoost", "OOF RMSLE": xgb_rmsle},
    {"Model": "ExtraTrees", "OOF RMSLE": et_rmsle},
    {"Model": "🔥 V17 FAST.AI SLSQP ENSEMBLE", "OOF RMSLE": masterpiece_rmsle},
]).sort_values(by="OOF RMSLE")

print("=" * 60)
print("FINAL V17 OOF PERFORMANCE COMPARISON (10-FOLD CV)")
print("=" * 60)
print(results_df.to_string(index=False))

plt.figure(figsize=(10, 5))
sns.barplot(data=results_df, x="OOF RMSLE", y="Model", palette="viridis")
plt.title("Notebook v17 Model OOF RMSLE Comparison (Lower is Better)")
plt.xlim(min(results_df["OOF RMSLE"]) * 0.98, max(results_df["OOF RMSLE"]) * 1.02)
plt.show()


<a id="12"></a>
# 12. Final Predictions & Export


In [ ]:
final_predictions = np.expm1(test_preds_opt_blend)
final_predictions = np.clip(final_predictions, a_min=0, a_max=None)

submission = sub_template.copy()
submission["TargetValue"] = final_predictions
submission.to_csv("submission.csv", index=False)

print("=" * 70)
print("FINAL V17 SUBMISSION VERIFICATION")
print("=" * 70)
print(f"Row count matches sample_submission : {len(submission) == len(sub_template)} ({len(submission):,} rows)")
print(f"Null values present                 : {submission['TargetValue'].isnull().sum()}")
print(f"Negative values present             : {(submission['TargetValue'] < 0).sum()}")
print("\nFirst 10 Predictions:")
print(submission.head(10))
print("\nSummary Statistics of Final Predictions:")
print(submission["TargetValue"].describe())

print("\n🎉 V17 FAST.AI PIPELINE COMPLETED SUCCESSFULLY!")


<a id="13"></a>
# 13. Conclusion & Key Takeaways

1. **Fast.ai Ordinal Categorical Encoding**: Preserved categorical hierarchy across all string variables without causing dimension blowup.
2. **Explicit Date Deconstruction**: Deconstructing sale dates into Year, Month, Day, DayOfWeek, DayOfYear, and ElapsedDays provided direct temporal splitting signals.
3. **Pure GBDT Tree Stacking**: Combining LightGBM, CatBoost, XGBoost, and ExtraTrees produced robust overall RMSLE precision.

